In [0]:
# Notebook lo first cell lo idi run cheyyi (DOCX library install cheyadaniki)
%pip install python-docx

In [0]:
import requests
import json
import os
import time
from docx import Document

# ==========================================
# 1. API Keys & Configurations
# ==========================================
JSEARCH_API_KEY = dbutils.secrets.get("jobs_automation", "jsearch")
NVIDIA_NIM_API_KEY = dbutils.secrets.get("jobs_automation", "nvidia_nemotron_3_ultra")

JOB_SEARCH_QUERY = "Data Engineer remote USA"
SAVE_DIRECTORY = "/Workspace/Users/maheshmahi1282@gmail.com/Jobs_Automation/Drafts/"
CACHE_FILE = os.path.join(SAVE_DIRECTORY, "cached_jobs.json")
RAW_FILE = os.path.join(SAVE_DIRECTORY, "raw_response.json")

os.makedirs(SAVE_DIRECTORY, exist_ok=True)

BASE_RESUME_TEXT = """
Name: Sindhu Priya Singamaneni
Title: Sr Data Engineer
Phone: +19722928093 | Email: sindhus8585@gmail.com
Summary: Data Engineer with 10 years of experience in designing, developing, and optimizing enterprise data integration, reporting, and analytics solutions.
Technical Skills: 
- Cloud Platforms: Azure (ADF, Databricks, Synapse), AWS (S3, EMR, Glue, Redshift)
- Big Data: Spark, PySpark, Hive
- Programming: Python, SQL, Java, Scala
"""

# ==========================================
# 2. Fetch Jobs (With Exact Nested Extraction)
# ==========================================
def fetch_jobs(query, num_pages=1):
    if os.path.exists(CACHE_FILE):
        print("♻️ SAVING API LIMIT: Loading jobs from local cache...")
        try:
            with open(CACHE_FILE, "r") as f:
                return json.load(f)
        except Exception as e:
            print("⚠️ Cache file corrupted, fetching fresh data...")

    print(f"🔍 Searching for jobs via JSearch API: {query}...")
    url = "https://jsearch.p.rapidapi.com/search-v2" 
    
    querystring = {"query": query, "page": str(num_pages), "num_pages": "1"}
    headers = {
        "x-rapidapi-key": JSEARCH_API_KEY,
        "x-rapidapi-host": "jsearch.p.rapidapi.com"
    }
    
    response = requests.get(url, headers=headers, params=querystring)
    print(f"📡 JSearch API Status Code: {response.status_code}")
    
    if response.status_code == 200:
        json_resp = response.json()
        
        with open(RAW_FILE, "w") as f:
            json.dump(json_resp, f, indent=4)
        print(f"💾 Raw API data saved to: {RAW_FILE}")
        
        jobs_list = []
        if isinstance(json_resp, dict):
            if 'data' in json_resp:
                inner_data = json_resp['data']
                if isinstance(inner_data, list):
                     jobs_list = inner_data
                elif isinstance(inner_data, dict) and 'jobs' in inner_data and isinstance(inner_data['jobs'], list):
                     jobs_list = inner_data['jobs']
            elif 'jobs' in json_resp and isinstance(json_resp['jobs'], list):
                 jobs_list = json_resp['jobs']
        elif isinstance(json_resp, list):
             jobs_list = json_resp
                    
        if len(jobs_list) > 0:
            with open(CACHE_FILE, "w") as f:
                json.dump(jobs_list, f)
            print("✅ Jobs successfully extracted and cached.")
            return jobs_list
        else:
            print("⚠️ Could not extract jobs from the response. Check raw_response.json.")
            return []
    else:
        print(f"❌ Error fetching jobs: {response.text}")
        return []

# ==========================================
# 3. Tailor Resume using NVIDIA (NEMOTRON-4)
# ==========================================
def tailor_resume(company_name, job_description):
    print(f"🧠 NVIDIA AI is Tailoring for {company_name}...")
    url = "https://integrate.api.nvidia.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {NVIDIA_NIM_API_KEY}",
        "Content-Type": "application/json"
    }
    
    prompt = f"""
    You are an expert technical recruiter. I have a Base Resume and a Job Description. 
    Rewrite the Professional Summary and highlight the Technical Skills from the Base Resume 
    so that it perfectly aligns with the Job Description. DO NOT invent fake experience. 
    Only use skills present in the Base Resume.

    Base Resume:
    {BASE_RESUME_TEXT}

    Job Description:
    {job_description}
    
    Return ONLY the tailored text (Professional Summary and Technical Skills). Do not include any greeting or conversational text.
    """
    
    payload = {
        # 💡 UPDATED MODEL: This will fix the 404 error
        "model": "nvidia/nemotron-4-340b-instruct", 
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.2,
        "max_tokens": 1024
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code == 200:
            return response.json()['choices'][0]['message']['content']
        else:
            print(f"❌ Error with NVIDIA API ({response.status_code}): {response.text}")
            return ""
    except Exception as e:
        print(f"❌ Python Error in NVIDIA Call: {str(e)}")
        return ""

# ==========================================
# 4. Pipeline Execution
# ==========================================
def run_pipeline():
    jobs = fetch_jobs(JOB_SEARCH_QUERY)
    
    if not jobs or len(jobs) == 0:
        print("🛑 Workflow stopped.")
        return

    print(f"\n✅ Validated: Found {len(jobs)} jobs. Processing the first 3...\n")
    
    for idx, job in enumerate(jobs[:3]):
        if not isinstance(job, dict):
            continue
            
        company_name = job.get('employer_name', f'Company_{idx}')
        job_title = job.get('job_title', 'Data Engineer')
        job_description = job.get('job_description', '')
        job_link = job.get('job_apply_link', 'No Apply Link Found')
        salary = job.get('job_salary_string', 'Salary Not Listed')
        
        print(f"🚀 Processing Job {idx+1}: {job_title} at {company_name}")
        
        if not job_description:
            print("⚠️ Skipping because Job Description is empty.")
            continue
            
        tailored_content = tailor_resume(company_name, job_description)
        
        if tailored_content:
            safe_company_name = "".join([c if c.isalnum() else "_" for c in company_name])
            file_name = f"{safe_company_name}_Resume.docx"
            file_path = os.path.join(SAVE_DIRECTORY, file_name)
            
            try:
                doc = Document()
                doc.add_heading('Sindhu Priya Singamaneni - Tailored Profile', 0)
                doc.add_paragraph(f"Target Company: {company_name}")
                doc.add_paragraph(f"Role: {job_title}")
                doc.add_paragraph(f"Estimated Salary: {salary}")
                doc.add_paragraph(f"Apply Here: {job_link}")
                
                doc.add_heading('Tailored Summary & Skills', level=1)
                doc.add_paragraph(tailored_content)
                
                doc.save(file_path)
                print(f"📄 DOCX saved successfully at: {file_path}\n")
            except Exception as e:
                print(f"❌ Error saving Word document: {str(e)}")
            
        # Small delay to respect NVIDIA limits
        time.sleep(2) 

# Start the pipeline
run_pipeline()

In [0]:
# ==========================================
# 3. Tailor Resume using NVIDIA (GUARANTEED MODEL)
# ==========================================
def tailor_resume(company_name, job_description):
    print(f"🧠 NVIDIA AI is Tailoring for {company_name}...")
    url = "https://integrate.api.nvidia.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {NVIDIA_NIM_API_KEY}",
        "Content-Type": "application/json"
    }
    
    prompt = f"""
    You are an expert technical recruiter. I have a Base Resume and a Job Description. 
    Rewrite the Professional Summary and highlight the Technical Skills from the Base Resume 
    so that it perfectly aligns with the Job Description. DO NOT invent fake experience. 
    Only use skills present in the Base Resume.

    Base Resume:
    {BASE_RESUME_TEXT}

    Job Description:
    {job_description}
    
    Return ONLY the tailored text (Professional Summary and Technical Skills). Do not include any greeting or conversational text.
    """
    
    payload = {
        # 💡 FIX: Changed to a universally available, highly powerful model on NVIDIA free tier
        "model": "meta/llama-3.1-70b-instruct", 
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.2,
        "max_tokens": 1024
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code == 200:
            return response.json()['choices'][0]['message']['content']
        else:
            print(f"❌ Error with NVIDIA API ({response.status_code}): {response.text}")
            return ""
    except Exception as e:
        print(f"❌ Python Error in NVIDIA Call: {str(e)}")
        return ""
tailor_resume("FlexBoard", "The application window is expected to reputed company on 12/25/25 reputed company may be removed earlier if the position is filled or if a sufficient number of applications are received. Job location preference Remote reputed company USA, RTP-NC and Austin-TX Meet reputed company Join the reputed company IT Data team, where innovation, automation, and reliability reputed company world-class business reputed company. reputed company delivers reputed company, secure, and high-performance platforms supporting reputed company's global data operations. We value a culture of reputed company improvement, collaboration, and technical reputed company, empowering team members to experiment and reputed company operational transformation. Your reputed company As a Data Operations (DevOps) Engineer, you will play a meaningful role in building, automating, and optimizing the infrastructure and processes that support the Corporate Functions - reputed company Data Warehouse. Your expertise will ensure the reliability, scalability, and reputed company of data platforms and pipelines across reputed company and on-reputed company environments. You'll collaborate closely with data engineers, software engineers, architects, and business partners to create robust solutions that accelerate data-driven decision-making at reputed company. Key Responsibilities \u2022 Automate deployment, monitoring, and management of data platforms and pipelines using industry-reputed company DevOps tools and reputed company processes. \u2022 Design, implement, and maintain CI/CD pipelines for ETL, analytics, and data applications (e.g., Informatica, DBT, Airflow, Python, Java). \u2022 Ensure high availability, performance, and reputed company of data systems in reputed company (reputed company, reputed company BigQuery, AWS/GCP/Azure) and hybrid environments. \u2022 reputed company infrastructure as reputed company (Terraform, CloudFormation, or similar) to provision and reputed company resources reputed company. \u2022 Implement observability and data reputed company monitoring using modern tools (e.g., reputed company, reputed company, Grafana, ELK). \u2022 Solve and reputed company issues in production data & workflows, collaborating with engineering and analytics teams for reputed company cause analysis and solution delivery. \u2022 reputed company automation and process improvement for data operations, reputed company upgrades, patching, and reputed company management. \u2022 Contribute to reputed company and compliance initiatives reputed company to data governance, reputed company controls, and audit readiness. \u2022 Mentor and support junior engineers, encouraging a culture of knowledge sharing and operational reputed company. Minimum Qualifications \u2022 Bachelor's or Master's degree in Computer Science, Engineering, or a reputed company field. \u2022 5-8 years of experience in DevOps, Data Operations, or reputed company IT engineering roles. \u2022 5-8 years of experience Proficiency with reputed company platforms (reputed company, AWS) and Working knowledge of ETL and workflow orchestration tools (Informatica, DBT, Airflow). \u2022 5-8 years of experience Hands-on experience with CI/CD tools (Jenkins, reputed company CI, etc.), scripting (Python, reputed company), and configuration management. \u2022 Working knowledge of ETL and workflow orchestration tools (Informatica, DBT, Airflow). \u2022 Familiarity with infrastructure as reputed company (Terraform, CloudFormation, etc.). \u2022 5+ years of experience with monitoring, logging, and alerting solutions (reputed company, Grafana, ELK, reputed company, etc.). \u2022 5-8 years of experience with containerization and orchestration (reputed company, reputed company). \u2022 5+ years of experience with Strong troubleshooting, incident management, and problem-solving skills. \u2022 Experience working in reputed company/Scrum teams and delivering in fast-paced environments. Preferred Qualifications \u2022 Experience supporting data warehouse or analytics platforms in reputed company settings. \u2022 Knowledge of data reputed company, reputed company, and governance frameworks. \u2022 Familiarity with automation tools and reputed company methodologies for operational efficiency. \u2022 Understanding of data pipelines, modeling, and analytics. \u2022 Excellent communication, collaboration, and documentation skills. Why reputed company? At reputed company, we're revolutionizing how data and infrastructure reputed company and protect organizations in the AI era - and reputed company. We've been innovating fearlessly for 40 years to create solutions that reputed company how humans and technology work together across the physical reputed company worlds. These solutions reputed company customers with unparalleled reputed company, visibility, and insights across the entire digital footprint. reputed company by the depth and breadth of our technology, we experiment and create meaningful solutions. Add to that our worldwide network of doers and experts, and you'll see that the opportunities to grow and build are reputed company. We work as reputed company, collaborating with reputed company to reputed company really big things happen on a global reputed company. Because our solutions are everywhere, our reputed company is everywhere. We are reputed company, and our reputed company starts with you. Message to applicants applying to work in the U.S. and/or Canada The starting salary reputed company posted for this position is $165,000.00 to $241,400.00 and reflects the projected salary reputed company for new hires in this position in U.S. and/or Canada locations, not including incentive compensation*, equity, or benefits. Individual pay is determined by the candidate's hiring location, market conditions, job-reputed company skillset, experience, qualifications, education, certifications, and/or training. The full salary reputed company for certain locations is listed below. For locations not listed below, the recruiter can reputed company more details about compensation for the role in your location during the hiring process. U.S. employees are reputed company benefits, subject to reputed company's plan eligibility rules, which include medical, dental and reputed company reputed company, a 401(k) plan with a reputed company matching contribution, reputed company parental leave, short and long-term disability coverage, and basic life reputed company. Please see the reputed company careers site to discover more benefits and perks. Employees may be eligible to receive reputed company of reputed company restricted stock reputed company, which reputed company following reputed company employment with reputed company for defined periods of time. U.S. employees are eligible for reputed company time away as described below, subject to reputed company's policies \u2022 10 reputed company holidays per full calendar year, plus 1 floating holiday for non-exempt employees \u2022 1 reputed company day off for employee's birthday, reputed company year-end holiday shutdown, and 4 reputed company days off for personal wellness determined by reputed company \u2022 Non-exempt employees receive 16 days of reputed company vacation time per full calendar year, accrued at reputed company of 4.92 hours per pay period for full-time employees \u2022 Exempt employees participate in reputed company's flexible vacation time off program, which has no defined limit on how much vacation time eligible employees may use (subject to availability and some business limitations) \u2022 80 hours of reputed company time off provided on hire date and reputed company January 1st thereafter, and up to 80 hours of unused reputed company time carried reputed company from one calendar year to the next \u2022 Additional reputed company time away may be requested to deal with critical or emergency issues for family members \u2022 Optional 10 reputed company days per full calendar year to volunteer For non-sales roles, employees are also eligible to earn annual bonuses subject to reputed company's policies. Employees on sales plans earn performance-reputed company incentive pay on top of their reputed company salary, which is split between quota and non-quota components, subject to the applicable reputed company plan. For quota-reputed company incentive pay, reputed company typically pays as follows \u2022 .75% of incentive reputed company for reputed company 1% of reputed company attainment up to 50% of quota; \u2022 1.5% of incentive reputed company for reputed company 1% of attainment between 50% and 75%; \u2022 1% of incentive reputed company for reputed company 1% of attainment between 75% and 100%; and \u2022 Once performance exceeds 100% attainment, incentive rates are at or above 1% for reputed company 1% of attainment with no cap on incentive compensation. For non-quota-reputed company sales performance reputed company such as strategic sales objectives, reputed company may pay 0% up to 125% of reputed company. reputed company sales plans do not have a minimum reputed company of performance for sales incentive compensation to be reputed company. The applicable full salary ranges for this position, by specific state, are listed below reputed company Metro Area $165,000.00 - $277,600.00 Non-Metro reputed company state & Washington state $146,700.00 - $247,000.00 \u2022 For quota-reputed company sales roles on reputed company's sales plan, the ranges provided in this posting include reputed company pay and sales reputed company incentive compensation combined. \u2022 * Employees in Illinois, whether exempt or non-exempt, will participate in a unique time off program to meet local requirements. Apply tot his job Apply To this Job")

In [0]:
import requests
import json
import os
import time
from docx import Document

# ==========================================
# 1. API Keys & Configurations
# ==========================================
JSEARCH_API_KEY = dbutils.secrets.get("jobs_automation", "jsearch")
NVIDIA_NIM_API_KEY = dbutils.secrets.get("jobs_automation", "nvidia_nemotron_3_ultra")

JOB_SEARCH_QUERY = "Data Engineer remote USA"
SAVE_DIRECTORY = "/Workspace/Users/maheshmahi1282@gmail.com/Jobs_Automation/Drafts/"
CACHE_FILE = os.path.join(SAVE_DIRECTORY, "cached_jobs.json")
RAW_FILE = os.path.join(SAVE_DIRECTORY, "raw_response.json")

os.makedirs(SAVE_DIRECTORY, exist_ok=True)

BASE_RESUME_TEXT = """
Name: Sindhu Priya Singamaneni
Title: Sr Data Engineer
Phone: +19722928093 | Email: sindhus8585@gmail.com
Summary: Data Engineer with 10 years of experience in designing, developing, and optimizing enterprise data integration, reporting, and analytics solutions.
Technical Skills: 
- Cloud Platforms: Azure (ADF, Databricks, Synapse), AWS (S3, EMR, Glue, Redshift)
- Big Data: Spark, PySpark, Hive
- Programming: Python, SQL, Java, Scala
"""

# ==========================================
# 2. Fetch Jobs (With Exact Nested Extraction)
# ==========================================
def fetch_jobs(query, num_pages=1):
    if os.path.exists(CACHE_FILE):
        print("♻️ SAVING API LIMIT: Loading jobs from local cache...")
        try:
            with open(CACHE_FILE, "r") as f:
                return json.load(f)
        except Exception as e:
            print("⚠️ Cache file corrupted, fetching fresh data...")

    print(f"🔍 Searching for jobs via JSearch API: {query}...")
    
    # Using the correct search-v2 endpoint as you mentioned
    url = "https://jsearch.p.rapidapi.com/search-v2" 
    
    querystring = {"query": query, "page": str(num_pages), "num_pages": "1"}
    headers = {
        "x-rapidapi-key": JSEARCH_API_KEY,
        "x-rapidapi-host": "jsearch.p.rapidapi.com"
    }
    
    response = requests.get(url, headers=headers, params=querystring)
    print(f"📡 JSearch API Status Code: {response.status_code}")
    
    if response.status_code == 200:
        json_resp = response.json()
        
        with open(RAW_FILE, "w") as f:
            json.dump(json_resp, f, indent=4)
        print(f"💾 Raw API data saved to: {RAW_FILE}")
        
        jobs_list = []
        
        # PERFECTED EXTRACTION LOGIC based on the JSON structure you provided
        if isinstance(json_resp, dict):
            if 'data' in json_resp:
                inner_data = json_resp['data']
                if isinstance(inner_data, list):
                     jobs_list = inner_data
                elif isinstance(inner_data, dict) and 'jobs' in inner_data and isinstance(inner_data['jobs'], list):
                     jobs_list = inner_data['jobs']
            elif 'jobs' in json_resp and isinstance(json_resp['jobs'], list):
                 jobs_list = json_resp['jobs']
        elif isinstance(json_resp, list):
             jobs_list = json_resp
                    
        if len(jobs_list) > 0:
            with open(CACHE_FILE, "w") as f:
                json.dump(jobs_list, f)
            print("✅ Jobs successfully extracted and cached.")
            return jobs_list
        else:
            print("⚠️ Could not extract jobs from the response. Check raw_response.json.")
            return []
    else:
        print(f"❌ Error fetching jobs: {response.text}")
        return []

# ==========================================
# 3. Tailor Resume using NVIDIA
# ==========================================
def tailor_resume(company_name, job_description):
    print(f"🧠 NVIDIA AI is Tailoring for {company_name}...")
    url = "https://integrate.api.nvidia.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {NVIDIA_NIM_API_KEY}",
        "Content-Type": "application/json"
    }
    
    prompt = f"""
    You are an expert technical recruiter. I have a Base Resume and a Job Description. 
    Rewrite the Professional Summary and highlight the Technical Skills from the Base Resume 
    so that it perfectly aligns with the Job Description. DO NOT invent fake experience. 
    Only use skills present in the Base Resume.

    Base Resume:
    {BASE_RESUME_TEXT}

    Job Description:
    {job_description}
    
    Return ONLY the tailored text (Professional Summary and Technical Skills).
    """
    
    payload = {
        "model": "nvidia/nemotron-3-ultra-550b", 
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.2,
        "max_tokens": 1024
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code == 200:
            return response.json()['choices'][0]['message']['content']
        else:
            print(f"❌ Error with NVIDIA API: {response.text}")
            return ""
    except Exception as e:
        print(f"❌ Python Error in NVIDIA Call: {str(e)}")
        return ""

# ==========================================
# 4. Pipeline Execution
# ==========================================
def run_pipeline():
    jobs = fetch_jobs(JOB_SEARCH_QUERY)
    
    if not jobs or len(jobs) == 0:
        print("🛑 Workflow stopped.")
        return

    print(f"\n✅ Validated: Found {len(jobs)} jobs. Processing the first 3...\n")
    
    for idx, job in enumerate(jobs[:3]):
        if not isinstance(job, dict):
            continue
            
        company_name = job.get('employer_name', f'Company_{idx}')
        job_title = job.get('job_title', 'Data Engineer')
        job_description = job.get('job_description', '')
        job_link = job.get('job_apply_link', 'No Apply Link Found')
        salary = job.get('job_salary_string', 'Salary Not Listed')
        
        print(f"🚀 Processing Job {idx+1}: {job_title} at {company_name}")
        
        if not job_description:
            print("⚠️ Skipping because Job Description is empty.")
            continue
            
        tailored_content = tailor_resume(company_name, job_description)
        
        if tailored_content:
            safe_company_name = "".join([c if c.isalnum() else "_" for c in company_name])
            file_name = f"{safe_company_name}_Resume.docx"
            file_path = os.path.join(SAVE_DIRECTORY, file_name)
            
            try:
                doc = Document()
                doc.add_heading('Sindhu Priya Singamaneni - Tailored Profile', 0)
                doc.add_paragraph(f"Target Company: {company_name}")
                doc.add_paragraph(f"Role: {job_title}")
                doc.add_paragraph(f"Estimated Salary: {salary}")
                doc.add_paragraph(f"Apply Here: {job_link}")
                
                doc.add_heading('Tailored Summary & Skills', level=1)
                doc.add_paragraph(tailored_content)
                
                doc.save(file_path)
                print(f"📄 DOCX saved successfully at: {file_path}\n")
            except Exception as e:
                print(f"❌ Error saving Word document: {str(e)}")
            
        time.sleep(2) 

# Start the pipeline
run_pipeline()

In [0]:
# ==========================================
# 3. Tailor Resume using NVIDIA (UPDATED MODEL)
# ==========================================
def tailor_resume(company_name, job_description):
    print(f"🧠 NVIDIA AI is Tailoring for {company_name}...")
    url = "https://integrate.api.nvidia.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {NVIDIA_NIM_API_KEY}",
        "Content-Type": "application/json"
    }
    
    prompt = f"""
    You are an expert technical recruiter. I have a Base Resume and a Job Description. 
    Rewrite the Professional Summary and highlight the Technical Skills from the Base Resume 
    so that it perfectly aligns with the Job Description. DO NOT invent fake experience. 
    Only use skills present in the Base Resume.

    Base Resume:
    {BASE_RESUME_TEXT}

    Job Description:
    {job_description}
    
    Return ONLY the tailored text (Professional Summary and Technical Skills). Do not include conversational text.
    """
    
    payload = {
        # 💡 FIX: Updated to the latest working Nemotron-4 model
        "model": "nvidia/nemotron-4-340b-instruct", 
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.2,
        "max_tokens": 1024
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code == 200:
            return response.json()['choices'][0]['message']['content']
        else:
            print(f"❌ Error with NVIDIA API ({response.status_code}): {response.text}")
            return ""
    except Exception as e:
        print(f"❌ Python Error in NVIDIA Call: {str(e)}")
        return ""

In [0]:
# ==========================================
# 2. Fetch Jobs using JSearch API (UPDATED)
# ==========================================
def fetch_jobs(query, num_pages=1):
    print(f"🔍 Searching for jobs: {query}")
    
    # NOTE: Oka vela Code Snippet lo URL vere laga unte, daantho deenni replace cheyyi
    url = "https://jsearch.p.rapidapi.com/search-v2" 
    
    querystring = {"query": query, "page": str(num_pages), "num_pages": "1"}
    headers = {
        "x-rapidapi-key": JSEARCH_API_KEY,
        "x-rapidapi-host": "jsearch.p.rapidapi.com"
    }
    
    try:
        response = requests.get(url, headers=headers, params=querystring)
        
        if response.status_code == 200:
            print("✅ Jobs fetched successfully!")
            return response.json().get('data', [])
        else:
            print(f"❌ Error API Response Code: {response.status_code}")
            print(f"❌ Error Details: {response.text}")
            print(f"👉 Tip: RapidAPI lo 'GET Job Search' click chesi exact URL ni check cheyyi.")
            return []
            
    except Exception as e:
        print(f"❌ Python Error: {str(e)}")
        return []
    
fetch_jobs("Data Engineer remote USA")

In [0]:
%sql
select * from list_Secrets()

In [0]:
import requests
import json
import os
import time
from docx import Document

# ==========================================
# 1. API Keys & Configurations
# ==========================================
JSEARCH_API_KEY = dbutils.secrets.get("jobs_automation", "jsearch")
NVIDIA_NIM_API_KEY = dbutils.secrets.get("jobs_automation", "nvidia_nemotron_3_ultra")

JOB_SEARCH_QUERY = "Data Engineer remote USA"
SAVE_DIRECTORY = "/Workspace/Users/maheshmahi1282@gmail.com/Jobs_Automation/Drafts/"

# Create directory in Databricks if it doesn't exist
os.makedirs(SAVE_DIRECTORY, exist_ok=True)

# Base details from the image provided
BASE_RESUME_TEXT = """
Name: Sindhu Priya Singamaneni
Title: Sr Data Engineer
Phone: +19722928093 | Email: sindhus8585@gmail.com
Summary: Data Engineer with 10 years of experience in designing, developing, and optimizing enterprise data integration, reporting, and analytics solutions. Strong expertise in building scalable ELT pipelines...
Technical Skills: 
- Cloud Platforms: Azure (ADF, Databricks, Synapse), AWS (S3, EMR, Glue, Redshift)
- Big Data: Spark, PySpark, Hive
- Programming: Python, SQL, Java, Scala
"""

# ==========================================
# 2. Fetch Jobs using JSearch API
# ==========================================
def fetch_jobs(query, num_pages=1):
    print(f"🔍 Searching for jobs: {query}...")
    url = "https://jsearch.p.rapidapi.com/search-v2" 
    querystring = {"query": query, "page": str(num_pages), "num_pages": "1"}
    headers = {
        "x-rapidapi-key": JSEARCH_API_KEY,
        "x-rapidapi-host": "jsearch.p.rapidapi.com"
    }
    
    try:
        response = requests.get(url, headers=headers, params=querystring)
        if response.status_code == 200:
            json_resp = response.json()
            
            # Safely extract the list of jobs to prevent KeyError
            if isinstance(json_resp, dict):
                if 'jobs' in json_resp:
                    return json_resp['jobs']
                elif 'data' in json_resp:
                    return json_resp['data']
            elif isinstance(json_resp, list):
                return json_resp
            return []
        else:
            print(f"❌ Error fetching jobs: {response.text}")
            return []
    except Exception as e:
        print(f"❌ Python Error during JSearch API call: {str(e)}")
        return []

# ==========================================
# 3. Tailor Resume using NVIDIA Nemotron
# ==========================================
def tailor_resume(company_name, job_description):
    print(f"🧠 Tailoring resume for {company_name} using NVIDIA Nemotron 3 Ultra...")
    
    url = "https://integrate.api.nvidia.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {NVIDIA_NIM_API_KEY}",
        "Content-Type": "application/json"
    }
    
    prompt = f"""
    You are an expert technical recruiter. I have a Base Resume and a Job Description. 
    Rewrite the Professional Summary and highlight the Technical Skills from the Base Resume 
    so that it perfectly aligns with the Job Description. DO NOT invent fake experience. 
    Only use skills present in the Base Resume.

    Base Resume:
    {BASE_RESUME_TEXT}

    Job Description:
    {job_description}
    
    Return ONLY the tailored text (Professional Summary and Technical Skills). Do not include any greeting or conversational text.
    """
    
    payload = {
        "model": "nvidia/nemotron-3-ultra-550b", 
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.2,
        "max_tokens": 1024
    }
    
    response = requests.post(url, headers=headers, json=payload)
    if response.status_code == 200:
        return response.json()['choices'][0]['message']['content']
    else:
        print(f"❌ Error with Nemotron API: {response.text}")
        return ""

# ==========================================
# 4. Main Execution (The Automation Loop)
# ==========================================
def run_pipeline():
    jobs = fetch_jobs(JOB_SEARCH_QUERY)
    
    if not jobs or not isinstance(jobs, list):
        print("⚠️ No valid jobs list found. Please check the API response.")
        return

    print(f"✅ Found {len(jobs)} jobs. Processing the first 3 jobs as a test run...\n")
    
    # Process only the first 3 jobs
    for idx, job in enumerate(jobs[:3]):
        company_name = job.get('employer_name', f'Company_{idx}')
        job_title = job.get('job_title', 'Data Engineer')
        job_description = job.get('job_description', '')
        job_link = job.get('job_apply_link', 'No Link')
        
        print(f"🚀 Processing Job {idx+1}: {job_title} at {company_name}")
        
        # Call NVIDIA API to tailor the content
        tailored_content = tailor_resume(company_name, job_description)
        
        if tailored_content:
            # Clean company name for the file name
            safe_company_name = "".join([c if c.isalnum() else "_" for c in company_name])
            file_name = f"{safe_company_name}_Tailored_Resume.docx"
            file_path = os.path.join(SAVE_DIRECTORY, file_name)
            
            # Generate DOCX File
            doc = Document()
            doc.add_heading('Sindhu Priya Singamaneni - Tailored Profile', 0)
            
            doc.add_paragraph(f"Target Company: {company_name}")
            doc.add_paragraph(f"Role: {job_title}")
            doc.add_paragraph(f"Job Link: {job_link}")
            
            doc.add_heading('AI Tailored Summary & Skills', level=1)
            doc.add_paragraph(tailored_content)
            
            # Save the file
            doc.save(file_path)
                
            print(f"📄 DOCX saved successfully at: {file_path}\n")
            
        # Small delay to avoid API rate limits
        time.sleep(2) 

# Start the pipeline
run_pipeline()

In [0]:
jsearch = "5e9d4dfe3amshd0f9720e536229cp15b4d9jsn18f0156846ec"
nvidia = "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"
gmail_pass = "ryju hmum pykz zfbl"